# Activity D: Agentic Navigation
### ISA Tutorial — CHIIR 2026

---

**Author:** Preetam Dammu, PhD Candidate, University of Washington · preetams@uw.edu  
**Please cite:** [Dammu & Roosta, CHIIR 2026](https://dl.acm.org/doi/abs/10.1145/3786304.3787893)

---

## Objective

Some questions cannot be answered by a single search or API call — the answer lives *inside* a page
that must be discovered by following links. This is **Agentic Navigation (Level 5)**.

| Level | Name | What the system needs |
|-------|------|-----------------------|
| 2 | Structured Lookup | Single value from a deterministic API |
| 3 | Grounded Closed-Corpus | Fixed document set (RAG) |
| 4 | Grounded Live-Corpus | Open web — one-shot search |
| **5** | **Agentic Navigation** | **Multi-step navigation: decide → act → observe → repeat** |

**What you will see:**
1. A search query for the latest GitHub commit returns stale/vague results — the answer is not at a top-level URL.
2. A ReAct agent navigates to the repo, finds the commit link, follows it, and summarizes the commit.
3. You replicate the same two steps in your browser and compare.

> **Runtime:** Default model is `microsoft/Phi-3.5-mini-instruct` (3.8 B params).  
> A **T4 GPU on Colab is strongly recommended** — this model needs ~4 GB VRAM.  
> On CPU it works but each generation step takes ~2 minutes.

In [ ]:
# ── Install dependencies ────────────────────────────────────────────────────
# transformers    : HuggingFace model loading & text generation
# torch           : tensor operations
# accelerate      : efficient model loading for Phi on GPU
# requests        : HTTP page fetching
# beautifulsoup4  : HTML parsing and link extraction
# ddgs            : DuckDuckGo search (to show why search alone fails)

!pip install -q transformers torch accelerate requests beautifulsoup4 ddgs

## Setting up the LLM

We use **Phi-3.5-mini-instruct** (Microsoft, 3.8 B parameters) for this activity.  
Unlike simpler activities, the ReAct loop requires the model to reliably emit structured output  
(`Thought:` / `Action:` / `Answer:`) across multiple turns — a capability that needs at least a 3B model.

> **CPU fallback:** Swap `MODEL_ID` to `"Qwen/Qwen2-0.5B-Instruct"`. Format adherence will be  
> less consistent, but it will run on any laptop. Expect occasional malformed steps.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# ── Model selection ──────────────────────────────────────────────────────────
MODEL_ID = "microsoft/Phi-3.5-mini-instruct"   # recommended — needs T4 GPU on Colab

# CPU fallback (less reliable ReAct format adherence):
# MODEL_ID = "Qwen/Qwen2-0.5B-Instruct"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {device}")
print(f"Loading {MODEL_ID} ...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto" if device == "cuda" else None,
)
if device == "cpu":
    model = model.to(device)
model.eval()
print("Model ready!")


def generate(prompt: str, max_new_tokens: int = 200) -> str:
    """Send a plain-text prompt to the LLM and return the response string."""
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

---

# Part 1 — Why Search Alone Fails Here

**Question:** *"What was the most recent commit to the HuggingFace Transformers repository about?"*

This question changes every few hours. A search engine returns:
- Cached blog posts about past releases
- The repo homepage — not the commit page
- Release announcements — not individual commits

The actual commit content lives at a URL like:  
`https://github.com/huggingface/transformers/commit/<sha>`  
...which you can only reach by *navigating* to the repo and *following* the commit link.

Let's confirm this by running a live search first.

In [ ]:
# ── 1.1  Show that a search query doesn't reach the commit content ─────────
from ddgs import DDGS

question = "What was the most recent commit to the HuggingFace Transformers repository about?"

print(f"Question: {question}\n")
print("Running web search...\n")

with DDGS() as ddgs:
    results = list(ddgs.text(
        "latest commit huggingface transformers github",
        max_results=4
    ))

for i, r in enumerate(results, 1):
    print(f"[{i}] {r['title']}")
    print(f"    {r['href']}")
    print(f"    {r['body'][:150]}...")
    print()

print("─" * 60)
print("Notice: none of these links point to an actual commit page.")
print("The commit content requires navigation — not just search.")

---

# Part 2 — Agentic Navigation with ReAct

## The ReAct Framework

**ReAct** (Reason + Act, [Yao et al. 2022](https://arxiv.org/abs/2210.03629)) interleaves reasoning
and acting in a simple loop:

```
Thought:      The agent explains what it knows and what to do next.
Action:       The agent calls a tool with arguments.
Observation:  The tool returns a result.
              ↑ repeat until...
Answer:       The agent gives the final answer.
```

The full trace — every Thought, Action, and Observation — is fed back into the prompt at each step,
so the model has complete context of everything it has done so far.

## Tools available to the agent

| Tool | What it does |
|------|-------------|
| `navigate(url)` | Fetch the page at `url`; return a text excerpt and the top links found |
| `find_link(keyword)` | Scan the current page for a link whose text or URL contains `keyword` |
| `read_page(query)` | Return the lines of the current page most relevant to `query` |

Two navigations are all we need:
1. `navigate` repo homepage → `find_link` commit SHA → URL of the commit page
2. `navigate` commit page → `read_page` → commit title + changed files → summarize

## Agent Memory

One practical challenge with ReAct: a small model may not know *how* to navigate a specific
site on its first attempt — it might use the wrong link text, call tools in the wrong order,
or forget to follow up after an Observation.

The fix is **episodic memory** — a record of a previously successful run on a similar task.
We include that trace in the system prompt so the agent can follow the same pattern.

This is the same idea behind systems like
[Reflexion](https://arxiv.org/abs/2303.11366) and [MemGPT](https://arxiv.org/abs/2310.08560):
store what worked, retrieve it when a new task is similar, let the model adapt rather than
reinvent.

> **Note:** The memory uses a *different* repo (HuggingFace `datasets`) so the agent cannot
> simply copy the answer — it must follow the pattern and apply it to `transformers`.

In [ ]:
# ── Agent memory: a recorded successful run on a similar task ───────────────
# Key lessons encoded in the memory:
#   1. Go to /commits/main — this page lists commits with clean, clickable links.
#      The repo homepage is noisier and harder to parse.
#   2. Use find_link('/commit/') to grab the first commit URL from that list.
#   3. Navigate to the commit page and read_page to extract the summary.
#
# The memory uses the ACTUAL latest transformers commit (3a3b59c) so the
# agent sees a real, complete trace — not a placeholder.
#
# Using the datasets repo as the example means the agent must
# adapt the pattern to transformers, not copy the answer.

AGENT_MEMORY = '''\
## Memory: past successful navigation

Question: What was the most recent commit to the HuggingFace datasets repository about?

Thought: I should navigate to the commits listing page for the datasets repo — it shows
  recent commits with clean links, which is easier to parse than the repo homepage.
Action: navigate("https://github.com/huggingface/datasets/commits/main")
Observation: Page loaded. Text includes commit entries. Links found include:
  [fix: correct arrow dtype mapping] -> https://github.com/huggingface/datasets/commit/9fa3a12
  [docs: update README] -> https://github.com/huggingface/datasets/commit/8bd21cc

Thought: I can see commit links. I will use find_link to get the first commit URL.
Action: find_link("/commit/")
Observation: Found: [fix: correct arrow dtype mapping in Arrow writer] ->
  https://github.com/huggingface/datasets/commit/9fa3a12

Thought: Now I navigate to that commit page to read the full details.
Action: navigate("https://github.com/huggingface/datasets/commit/9fa3a12")
Observation: Page loaded. Commit title: 'fix: correct arrow dtype mapping in Arrow writer'.
  Author: lhoestq. Files changed: src/datasets/arrow_writer.py, tests/test_arrow_writer.py

Thought: I have the commit title and changed files. I can now write the answer.
Answer: The most recent commit fixed an incorrect dtype mapping in the Arrow writer
  (arrow_writer.py) and updated the corresponding tests.
'''

print("Agent memory loaded — navigation strategy: /commits/main -> find_link -> commit page")
print(f"Memory length: {len(AGENT_MEMORY)} characters")

In [ ]:
import requests
import re
from bs4 import BeautifulSoup

_UA = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/123.0.0.0 Safari/537.36"
)

# ── Agent state (the browser tab) ───────────────────────────────────────────
_state = {"url": None, "soup": None, "text": None}


def navigate(url: str) -> str:
    """Fetch url, store the parsed page, return text excerpt + top links."""
    resp = requests.get(url.strip(), headers={"User-Agent": _UA}, timeout=15)
    soup = BeautifulSoup(resp.text, "html.parser")
    _state.update({"url": url, "soup": soup, "text": soup.get_text(" ", strip=True)})

    excerpt = _state["text"][:1500]

    links = []
    for a in soup.find_all("a", href=True):
        text = a.get_text(strip=True)
        href = a["href"]
        if text and len(text) < 80:
            full = href if href.startswith("http") else f"https://github.com{href}"
            links.append(f"  [{text}] -> {full}")

    link_block = "\n".join(links[:20])
    return f"Navigated to: {url}\n\nPage excerpt:\n{excerpt}\n\nLinks found:\n{link_block}"


def find_link(keyword: str) -> str:
    """Find the first link on the current page matching keyword. Returns the full URL."""
    if _state["soup"] is None:
        return "Error: no page loaded yet. Call navigate(url) first."

    keyword_lower = keyword.lower()
    for a in _state["soup"].find_all("a", href=True):
        text = a.get_text(strip=True).lower()
        href = a["href"].lower()
        if keyword_lower in text or keyword_lower in href:
            full = a["href"] if a["href"].startswith("http") else f"https://github.com{a['href']}"
            return f"Found: [{a.get_text(strip=True)}] -> {full}"

    return f"No link containing '{keyword}' found on current page."


def read_page(query: str) -> str:
    """Return the lines of the current page most relevant to query (keyword overlap)."""
    if _state["text"] is None:
        return "Error: no page loaded yet. Call navigate(url) first."

    query_words = set(query.lower().split())
    lines = [ln.strip() for ln in _state["text"].splitlines() if ln.strip()]
    scored = sorted(lines, key=lambda ln: sum(1 for w in query_words if w in ln.lower()), reverse=True)
    top = [ln for ln in scored if any(w in ln.lower() for w in query_words)][:10]
    return "\n".join(top) if top else "No relevant lines found."


print("Tools defined: navigate(), find_link(), read_page()")

In [ ]:
import re

TOOLS = {
    "navigate":  navigate,
    "find_link": find_link,
    "read_page": read_page,
}

# The system prompt includes the agent memory so the model knows the correct
# action sequence, link keywords, and output format before it even starts.
REACT_SYSTEM_PROMPT = (
    "You are a web navigation agent. Answer the question by using the tools below.\n\n"
    "At EVERY step output EXACTLY ONE of these formats — nothing else:\n\n"
    "  Thought: <one sentence of reasoning>\n"
    "  Action: tool_name(\"argument\")\n\n"
    "OR when you have the final answer:\n\n"
    "  Answer: <your answer>\n\n"
    "Available tools:\n"
    "  navigate(\"url\")       - fetch a web page; returns text excerpt and links\n"
    "  find_link(\"keyword\")  - find a link on the current page matching keyword\n"
    "  read_page(\"query\")    - return the most relevant lines from the current page\n\n"
    "Rules:\n"
    "- Always write a Thought before every Action.\n"
    "- After each Observation, write another Thought before the next Action.\n"
    "- Call only ONE tool per step.\n"
    "- When you have enough information, write Answer: and stop.\n\n"
    + AGENT_MEMORY
)


def call_tool(action_str: str) -> str:
    """Parse 'tool_name("arg")' and call the matching tool."""
    match = re.match(r'(\w+)\s*\(\s*[\"\'](.*?)[\"\']\s*\)', action_str.strip())
    if not match:
        return f"Parse error: could not parse action: {action_str!r}"
    tool_name, arg = match.group(1), match.group(2)
    if tool_name not in TOOLS:
        return f"Unknown tool: {tool_name}. Available: {list(TOOLS)}"
    return TOOLS[tool_name](arg)


def react_agent(question: str, max_steps: int = 6) -> str:
    """
    Run the ReAct loop for up to max_steps steps.
    At each step: generate Thought+Action, call the tool, append Observation, repeat.
    The AGENT_MEMORY in the system prompt guides the model toward the correct pattern.
    """
    trace = f"Question: {question}\n"
    print(trace)

    for step in range(1, max_steps + 1):
        print(f"{'─' * 55}")
        print(f"Step {step}")
        print(f"{'─' * 55}")

        prompt = REACT_SYSTEM_PROMPT + "\n\n" + trace
        output = generate(prompt, max_new_tokens=120).strip()

        # Keep only the first reasoning block (prevent rambling)
        output = "\n".join(output.splitlines()[:5]).strip()
        print(output)
        trace += output + "\n"

        if "Answer:" in output:
            break

        action_match = re.search(r"Action:\s*(.+)", output)
        if not action_match:
            print("(No Action found — stopping)")
            break

        action_str = action_match.group(1).strip()
        print(f"\nCalling: {action_str}")
        observation = call_tool(action_str)

        obs_short = observation[:1200] + ("..." if len(observation) > 1200 else "")
        obs_block = f"Observation: {obs_short}"
        print(f"\n{obs_block[:400]}...\n")
        trace += obs_block + "\n"

    return trace


print("ReAct agent ready (memory-augmented).")

In [ ]:
# ── Run the agent ────────────────────────────────────────────────────────────
# The agent memory tells it to:
#   1. Navigate to /commits/main (not the repo homepage)
#   2. Use find_link('/commit/') to grab the latest commit URL
#   3. Navigate to the commit page and read_page to extract the summary
#
# The actual latest commit is [docs] model cards (#44837) at:
# https://github.com/huggingface/transformers/commit/3a3b59cb1a7c0238c8d1072e35d3879c5faff48e

final_trace = react_agent(
    "What was the most recent commit to the HuggingFace Transformers repository about?",
    max_steps=6,
)

print("\n" + "=" * 60)
print("FULL AGENT TRACE:")
print("=" * 60)
print(final_trace)

---

# Part 3 — Try It Yourself in the Browser

The agent used two navigations. You can replicate them in ~30 seconds:

**Step 1 — Open the commits listing**
> Go to [https://github.com/huggingface/transformers/commits/main](https://github.com/huggingface/transformers/commits/main)

You will see a list of recent commits, each with a title and a short SHA link on the right.
Click the title of the top commit.

**Step 2 — Read the commit page**
> You are now on a page like:
> `https://github.com/huggingface/transformers/commit/3a3b59cb1a7c0238c8d1072e35d3879c5faff48e`

The commit title appears at the top: **[docs] model cards (#44837)**.  
Below it, 12 files changed — all model documentation in `docs/source/en/model_doc/`.

**Compare with the agent trace:**
- Did the agent navigate to `/commits/main` first (as the memory instructed)?
- Did it use `find_link('/commit/')` to discover the URL without being told it?
- Did it correctly summarize the commit as a documentation update?

The agent discovered the commit URL by reading and following links — exactly what you just did.

### Observations

| | Web search (Level 4) | Agent navigation (Level 5) |
|---|---|---|
| Entry point | Search query | Known starting URL |
| Steps | 1 (search → read snippet) | 2+ (navigate → follow link → read page) |
| What it can reach | Top-level indexed pages | Any linked page, regardless of search rank |
| Answer currency | Depends on search index | Live — fetched at runtime |
| Failure mode | Stale/vague snippets | Page structure changes (fragile selectors) |

**Key insight:** The agent doesn't know the commit URL ahead of time — it *discovers* it by reading
the page, exactly like a human would. This is the defining property of Level 5.

---

# Recap — Complexity Ladder

| Level | Name | Source | Key challenge |
|-------|------|--------|---------------|
| 2 | Structured Lookup | Deterministic API | Schema design |
| 3 | Grounded Closed-Corpus | Fixed corpus (RAG) | Retrieval quality |
| 4 | Grounded Live-Corpus | Open web (search) | Source credibility |
| **5** | **Agentic Navigation** | **Any linked page** | **Planning, loop termination, fragile selectors** |

**Level 5 evaluation requires:**
- **Faithfulness** — does the answer match the actual page content?
- **Efficiency** — did the agent take the minimum number of steps?
- **Robustness** — does it still work when the page structure changes?

**Level 5 → Level 6 (Corpus Sensemaking):** instead of navigating to answer *one* question,
the agent builds structured understanding across *many* pages — clustering, ranking, and synthesizing at scale.